In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from pathlib import Path
import time
from missing_citation_retriever.missing_citation_retriever import MissingCitationRetriever

import nest_asyncio


nest_asyncio.apply()


# set the style for the plots
plt.style.use('seaborn-v0_8-paper')

OUT_DIR = 'figures/'
# Create the output directory if it doesn't exist
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

# GPT4o for citing sentence detection

In [ ]:
# GPT4o performance for missing reference detection by percentange of references found for three different papers
with_reference_markers = [0.16, 0.51, 0.05]
no_reference_markers = [0.05, 0.2, 0.03]

# Create a bar chart with large font size labels
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(['Paper 1 (new)', 'Paper 2 (old, popular)', 'Paper 3 (old, unpopular)'], with_reference_markers, label='With Reference Markers', alpha=0.7)
ax.bar(['Paper 1 (new)', 'Paper 2 (old, popular)', 'Paper 3 (old, unpopular)'], no_reference_markers, label='No Reference Markers', alpha=0.7)

# Add labels and title
ax.set_ylabel('Percentage of References Found', fontsize=16)
ax.set_title('GPT4o Performance for Missing Reference Detection', fontsize=18)
ax.legend(fontsize=14)
# Y-axis limits
ax.set_ylim(0, 1.0)
# bigger font size for better readability
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

# Show the plot
plt.tight_layout()
plt.show()

# Save the figure
fig.savefig(OUT_DIR + 'gpt4o_performance_missing_reference_detection.png', dpi=300, bbox_inches='tight', metadata={'CreationDate': None})

# Gemini 2.0 Flash Fine-Tuned for Citing Sentence Detection

## Training

In [ ]:
training_acc_path = Path('../data/gemini_citsent_train/Accuracy.csv')

# Load the training accuracy data
training_acc = pd.read_csv(training_acc_path)

training_acc.describe()

In [ ]:
# Plot the training accuracy for both train and validation sets
# Keep in mind that not every epoch has a validation accuracy, so we will just draw the train accuracy for every epoch and validation accuracy only for those epochs where it was calculated.

# Convert 'undefined' values to NaN for proper handling
training_acc['validation'] = pd.to_numeric(training_acc['validation'], errors='coerce')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(training_acc['TimeSeries ID'], training_acc['train'], label='Train Accuracy', marker='x', linestyle='-')
# Plot validation accuracy with dropna=False to ensure continuous line between valid points
valid_mask = ~training_acc['validation'].isna()
if valid_mask.any():
    # Get only the rows with validation data
    valid_points = training_acc[valid_mask]
    ax.plot(valid_points['TimeSeries ID'], valid_points['validation'],
            label='Validation Accuracy', marker='x', linestyle='-')
# Add labels and title
ax.set_xlabel('Step', fontsize=18)
ax.set_ylabel('Accuracy', fontsize=18)
ax.set_title('Training and Validation Accuracy Over Steps', fontsize=18)

# Add a legend
ax.legend(fontsize=16)
# Set y-axis limits
ax.set_ylim(0.75, 1.0)
# Set font size for x and y ticks
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
# Show the plot
plt.tight_layout()
plt.show()
# Save the figure
fig.savefig(OUT_DIR + 'gemini2_flash_training_validation_accuracy.png', dpi=300, bbox_inches='tight', metadata={'CreationDate': None})

In [ ]:
training_loss_path = Path('../data/gemini_citsent_train/Loss.csv')
# Load the training loss data
training_loss = pd.read_csv(training_loss_path)
training_loss.describe()

In [ ]:
# Plot the training loss for both train and validation sets
# Keep in mind that not every epoch has a validation loss, so we will handle it similarly to accuracy

# Convert 'undefined' values to NaN for proper handling
training_loss['validation'] = pd.to_numeric(training_loss['validation'], errors='coerce')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(training_loss['TimeSeries ID'], training_loss['train'], label='Train Loss', marker='x', linestyle='-')
# Plot validation loss with dropna=False to ensure continuous line between valid points
valid_mask = ~training_loss['validation'].isna()
if valid_mask.any():
    # Get only the rows with validation data
    valid_points = training_loss[valid_mask]
    ax.plot(valid_points['TimeSeries ID'], valid_points['validation'],
            label='Validation Loss', marker='x', linestyle='-')
# Add labels and title
ax.set_xlabel('Step', fontsize=18)
ax.set_ylabel('Loss', fontsize=18)
ax.set_title('Training and Validation Loss Over Steps', fontsize=18)

# Add a legend
ax.legend(fontsize=16)
# Set font size for x and y ticks
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
# Show the plot
plt.tight_layout()
plt.show()
# Save the figure
fig.savefig(OUT_DIR + 'gemini2_flash_training_validation_loss.png', dpi=300, bbox_inches='tight', metadata={'CreationDate': None})


## Confusion Matrix

In [ ]:

cm = [[71703, 9243],
      [13633, 23606]]

# convert to numpy array for plotting
cm = np.array(cm)

# Plot the confusion matrix with numbers on squares
fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.matshow(cm)
# Add color bar
plt.colorbar(cax)
# Add text annotations in the squares
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(x=j, y=i, s=cm[i, j], va='center', ha='center', fontsize=16)
# Add labels and title
ax.set_title('Confusion Matrix for Gemini 2.0 Flash Fine-Tuned Model', fontsize=18)
ax.set_ylabel('True label', fontsize=16)
ax.set_xlabel('Predicted label', fontsize=16)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Non-citing', 'citing'], fontsize=14)
ax.set_yticks([0, 1])
ax.set_yticklabels(['Non-citing', 'citing'], fontsize=14)
plt.tight_layout()
plt.show()
# Save the figure
fig.savefig(OUT_DIR + 'gemini-citsent-confusion-matrix.png', dpi=300, bbox_inches='tight', metadata={'CreationDate': None})


# Missing citation retriever average time

In [2]:
retriever = MissingCitationRetriever()

PAPER_FOLDER = Path('../data/loose_pdfs/')

# Get the list of all PDF files in the folder
pdf_files = list(PAPER_FOLDER.glob('*.pdf'))

2025-06-19 19:20:28,082 - INFO - PyTorch version 2.5.1 available.
2025-06-19 19:20:28,438 - INFO - Load pretrained SentenceTransformer: intfloat/multilingual-e5-large-instruct
2025-06-19 19:20:36,176 - INFO - Loaded embedding model intfloat/multilingual-e5-large-instruct on device cuda
2025-06-19 19:20:36,186 - INFO - GET http://localhost:9200/ [status:200 duration:0.010s]
2025-06-19 19:20:36,205 - INFO - HEAD http://localhost:9200/ [status:200 duration:0.018s]
2025-06-19 19:20:36,278 - INFO - Using Gemini for reranking


In [3]:
# Calculate the average time for each PDF file
times = []
for pdf_file in pdf_files:
    # Measure the time taken to retrieve missing citations
    start_time = time.time()
    await retriever.check_paper(pdf_file)
    time_taken = time.time() - start_time
    times.append(time_taken)

2025-06-19 19:20:44,260 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:44,600 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:45,022 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:45,230 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:45,567 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:46,045 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:46,387 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:46,530 - INFO - AFC is enabled with max remote calls: 10.


Rate limit hit, backing off for 0.62 seconds (retry 1/25)...
Rate limit hit, backing off for 0.91 seconds (retry 1/25)...
Rate limit hit, backing off for 0.65 seconds (retry 1/25)...


2025-06-19 19:20:46,986 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:47,129 - INFO - AFC is enabled with max remote calls: 10.


Rate limit hit, backing off for 0.60 seconds (retry 1/25)...
Rate limit hit, backing off for 0.88 seconds (retry 1/25)...


2025-06-19 19:20:47,420 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:47,595 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:47,755 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:47,756 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:48,100 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:48,135 - INFO - HTTP Request: POST https://europe-west9-aiplatform.googleapis.com/v1beta1/projects/438747908796/locations/europe-west9/endpoints/3205177550036795392:generateContent "HTTP/1.1 200 OK"


...

2025-06-19 19:20:48,154 - INFO - AFC remote call 1 is done.
2025-06-19 19:20:48,155 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:48,246 - INFO - HTTP Request: POST https://europe-west9-aiplatform.googleapis.com/v1beta1/projects/438747908796/locations/europe-west9/endpoints/3205177550036795392:generateContent "HTTP/1.1 200 OK"
2025-06-19 19:20:48,248 - INFO - AFC remote call 1 is done.
2025-06-19 19:20:48,249 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:48,275 - INFO - HTTP Request: POST https://europe-west9-aiplatform.googleapis.com/v1beta1/projects/438747908796/locations/europe-west9/endpoints/3205177550036795392:generateContent "HTTP/1.1 200 OK"
2025-06-19 19:20:48,288 - INFO - AFC remote call 1 is done.
2025-06-19 19:20:48,289 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:48,290 - INFO - HTTP Request: POST https://europe-west9-aiplatform.googleapis.com/v1beta1/projects/438747908796/locations/europe-west9/endpoints/3

Rate limit hit, backing off for 4.49 seconds (retry 2/25)...


2025-06-19 19:20:48,799 - INFO - HTTP Request: POST https://europe-west9-aiplatform.googleapis.com/v1beta1/projects/438747908796/locations/europe-west9/endpoints/3205177550036795392:generateContent "HTTP/1.1 200 OK"
2025-06-19 19:20:48,814 - INFO - AFC remote call 1 is done.
2025-06-19 19:20:48,814 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:48,862 - INFO - HTTP Request: POST https://europe-west9-aiplatform.googleapis.com/v1beta1/projects/438747908796/locations/europe-west9/endpoints/3205177550036795392:generateContent "HTTP/1.1 200 OK"
2025-06-19 19:20:48,865 - INFO - AFC remote call 1 is done.
2025-06-19 19:20:48,992 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:20:49,004 - INFO - HTTP Request: POST https://europe-west9-aiplatform.googleapis.com/v1beta1/projects/438747908796/locations/europe-west9/endpoints/3205177550036795392:generateContent "HTTP/1.1 200 OK"
2025-06-19 19:20:49,022 - INFO - AFC remote call 1 is done.
2025-06-19 19:20:49,06

2025-06-19 19:22:09,964 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:22:10,025 - INFO - HTTP Request: POST https://europe-west9-aiplatform.googleapis.com/v1beta1/projects/438747908796/locations/europe-west9/endpoints/3205177550036795392:generateContent "HTTP/1.1 200 OK"
2025-06-19 19:22:10,026 - INFO - AFC remote call 1 is done.
2025-06-19 19:22:10,102 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:22:10,348 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:22:10,571 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:22:10,579 - INFO - HTTP Request: POST https://europe-west9-aiplatform.googleapis.com/v1beta1/projects/438747908796/locations/europe-west9/endpoints/3205177550036795392:generateContent "HTTP/1.1 200 OK"
2025-06-19 19:22:10,588 - INFO - AFC remote call 1 is done.
2025-06-19 19:22:10,588 - INFO - HTTP Request: POST https://europe-west9-aiplatform.googleapis.com/v1beta1/projects/438747908796/locations/europe-we

2025-06-19 19:24:55,256 - INFO - HTTP Request: POST https://europe-west9-aiplatform.googleapis.com/v1beta1/projects/438747908796/locations/europe-west9/endpoints/3205177550036795392:generateContent "HTTP/1.1 200 OK"
2025-06-19 19:24:55,257 - INFO - AFC remote call 1 is done.
2025-06-19 19:24:55,436 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:24:55,613 - INFO - AFC is enabled with max remote calls: 10.
2025-06-19 19:24:55,627 - INFO - HTTP Request: POST https://europe-west9-aiplatform.googleapis.com/v1beta1/projects/438747908796/locations/europe-west9/endpoints/3205177550036795392:generateContent "HTTP/1.1 200 OK"
2025-06-19 19:24:55,629 - INFO - AFC remote call 1 is done.
2025-06-19 19:24:55,818 - INFO - HTTP Request: POST https://europe-west9-aiplatform.googleapis.com/v1beta1/projects/438747908796/locations/europe-west9/endpoints/3205177550036795392:generateContent "HTTP/1.1 200 OK"
2025-06-19 19:24:55,850 - INFO - AFC remote call 1 is done.
2025-06-19 19:24:56,09

In [4]:
# Calculate the average time
average_time = np.mean(times)

print(f'Average time for missing citation retrieval: {average_time:.2f} seconds')

# Standard deviation of the time
std_time = np.std(times)

print(f'Standard deviation of time: {std_time:.2f} seconds')

Average time for missing citation retrieval: 39.50 seconds
Standard deviation of time: 17.44 seconds
